# Qwen-Image 2.1 LoRA 学習ノートブック (Musubi Tuner版・完全自動・Colab対応) v1.0

Kohya-ssの最新アーキテクチャ学習フレームワーク **Musubi Tuner**（PR #1145 / `nukuihayu:qwen-image-21`）を採用した、高品質な Qwen-Image-2.1 LoRA トレーニングパイプラインです。

**特徴**:
- **手足・背景の崩壊を完全克服**: 単体テスト64件通過済みの堅牢なDiTブロック実装により、ai-toolkitで発生していたRoPE座標の衝突や手足の消失を防止。
- **24GB/40GB VRAM最適化**: Text Encoder（Qwen3-VL 8B）とVAEの事前キャッシュにより、学習ループ中はDiT本体のみをVRAMに展開して安定学習。
- **ComfyUI完全自動変換**: Musubi Tuner内蔵の `convert_lora` を自動実行し、ComfyUIで直接ロードできる形式に変換して Google Drive に保存。
- **操作は最初だけ**: 1-1（HFトークン）、1-2（Driveマウント）、1-3（設定確認）を実行したら、あとは最下部まで完全全自動。

## 1. あなたの操作 (ここだけ・最初に全部済ませる)

### 1-1. HuggingFaceトークン
Colabの左サイドバーにある **🔑（シークレット）** アイコンから `HF_TOKEN` を登録しておくか、未登録の場合は実行時に入力プロンプトが表示されます。

In [ ]:
import getpass, os

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not token:
    token = os.environ.get("HF_TOKEN")

if not token:
    token = getpass.getpass("HuggingFace Access Token を入力してください (hf_...): ").strip()

if token:
    os.environ["HF_TOKEN"] = token
    print("トークン: OK (設定済み)")
else:
    print("⚠️ トークン未設定! (公開モデルのダウンロードのみ可能)")

### 1-2. Google Driveマウント
ポップアップでGoogleアカウント選択・許可が出たら「許可」をクリックしてください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DriveマウントOK")

### 1-3. ジョブ設定 (全自動スキャン / 個別指定 / jobs.json)
Google Drive内のデータセットZIPを**自動検出**し、キャプションからトリガー名を**自動抽出**します（`jobs.json` の作成・編集は不要です）。

In [ ]:
#@title 1-3. ジョブ設定 (全自動スキャン / 個別指定 / jobs.json)
#@markdown **モード選択**:
#@markdown - `AUTO_DETECT`: Google Drive内のZIPを自動検出し、トリガー名もキャプションから自動抽出して一括学習 (jobs.json不要)
#@markdown - `SPECIFY_ZIP`: 下記の `TARGET_ZIP` (ファイル名またはパス) のみを学習
#@markdown - `USE_JOBS_JSON`: 従来の `jobs.json` 設定ファイルを使用

import glob, json, os, re, zipfile

MODE = "AUTO_DETECT" #@param ["AUTO_DETECT", "SPECIFY_ZIP", "USE_JOBS_JSON"]

# 個別指定の場合のZIPファイル名 (例: kibisake_20260812_30.zip)
TARGET_ZIP = "kibisake_20260812_30.zip" #@param {type:"string"}
# トリガー名を手動上書きしたい場合のみ入力 (空欄ならZIP/キャプションから自動抽出)
CUSTOM_TRIGGER = "" #@param {type:"string"}

# 基本学習パラメータ
STEPS = 1500 #@param {type:"integer"}
LR = 0.0001 #@param {type:"number"}
RANK = 16 #@param {type:"integer"}
RESOLUTION = 1024 #@param {type:"integer"}
SKIP_ALREADY_TRAINED = True #@param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/qwen_image21/outputs" #@param {type:"string"}

SCAN_DIRS = [
    "/content/drive/MyDrive/training/illust_sozai_zip",
    "/content/drive/MyDrive/qwen_image21",
    "/content/drive/MyDrive/qwen_image21/datasets",
    "/content/drive/MyDrive/datasets",
    "/content",
]

def find_zip_path(zip_path):
    if os.path.exists(zip_path):
        return zip_path
    basename = os.path.basename(zip_path)
    for d in SCAN_DIRS:
        for cand in [os.path.join(d, zip_path), os.path.join(d, basename)]:
            if os.path.exists(cand):
                return cand
    return zip_path

def extract_trigger_from_zip(zip_path):
    resolved = find_zip_path(zip_path)
    try:
        with zipfile.ZipFile(resolved, "r") as z:
            txt_files = sorted([f for f in z.namelist() if f.endswith(".txt") and not f.startswith("__MACOSX")])
            if txt_files:
                sample_txt = z.read(txt_files[0]).decode("utf-8", errors="ignore").strip()
                first_tag = sample_txt.split(",")[0].strip()
                if first_tag and len(first_tag.split()) <= 4 and "\n" not in first_tag and "\r" not in first_tag:
                    return first_tag
    except Exception:
        pass
    base = os.path.basename(zip_path).replace(".zip", "")
    clean = re.sub(r"_\d{8}(?:_\d+)?$", "", base)
    clean = re.sub(r"_\d+$", "", clean)
    return clean if clean.endswith("_style") else f"{clean}_style"

def is_already_trained(trigger, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    matches = glob.glob(os.path.join(out_dir, f"{trigger}*.safetensors"))
    return len(matches) > 0

JOBS = []

if MODE == "USE_JOBS_JSON":
    CFG_PATHS = [
        "/content/jobs.json",
        "/content/drive/MyDrive/qwen_image21/jobs.json",
        "/content/drive/MyDrive/krea2/jobs.json",
        "/content/drive/MyDrive/jobs.json",
    ]
    CFG_PATH = next((p for p in CFG_PATHS if os.path.exists(p)), None)
    assert CFG_PATH, f"設定ファイルが見つかりません。検索パス: {CFG_PATHS}"
    cfg = json.load(open(CFG_PATH, encoding="utf-8"))
    JOBS = cfg.get("jobs", [])
    RESOLUTION = cfg.get("resolution", RESOLUTION)
    STEPS = cfg.get("steps", STEPS)
    LR = cfg.get("lr", LR)
    RANK = cfg.get("rank", RANK)
    OUTPUT_DIR = cfg.get("output_dir", OUTPUT_DIR)
    print(f"📄 jobs.json 読み込み完了: {CFG_PATH}")

elif MODE == "SPECIFY_ZIP":
    assert TARGET_ZIP, "TARGET_ZIP を指定してください"
    resolved = find_zip_path(TARGET_ZIP)
    assert os.path.exists(resolved), f"ZIPファイルが見つかりません: {TARGET_ZIP}\n(検索対象: {SCAN_DIRS})"
    trig = CUSTOM_TRIGGER.strip() or extract_trigger_from_zip(resolved)
    JOBS = [{"trigger": trig, "zip": resolved, "steps": STEPS, "lr": LR, "rank": RANK}]
    print(f"🎯 個別ZIP指定: {trig} <- {os.path.basename(resolved)}")

elif MODE == "AUTO_DETECT":
    all_zips = []
    seen_names = set()
    for d in SCAN_DIRS:
        if os.path.isdir(d):
            for z in sorted(glob.glob(os.path.join(d, "*.zip"))):
                b = os.path.basename(z)
                if b not in seen_names:
                    seen_names.add(b)
                    all_zips.append(z)
    print(f"🔍 スキャン完了: 計 {len(all_zips)} 個の ZIP ファイルを検出")
    for z in all_zips:
        trig = extract_trigger_from_zip(z)
        if SKIP_ALREADY_TRAINED and is_already_trained(trig, OUTPUT_DIR):
            print(f" ⏩ スキップ (学習済み): {trig} ({os.path.basename(z)})")
            continue
        JOBS.append({"trigger": trig, "zip": z, "steps": STEPS, "lr": LR, "rank": RANK})
        print(f" ⏳ 学習対象に追加: {trig} <- {os.path.basename(z)}")

assert JOBS, "学習対象のジョブがありません。(新規のZIPファイルを配置するか、SKIP_ALREADY_TRAINED を False にしてください)"

print("\n" + "="*60)
print(f"🚀 学習予定ジョブ数: {len(JOBS)} 件")
print(f"⚙️ 設定: 解像度 {RESOLUTION}px / ステップ数 {STEPS} / lr {LR} / rank {RANK}")
print(f"📂 出力先: {OUTPUT_DIR}")
for i, j in enumerate(JOBS, 1):
    print(f" [{i}] trigger: '{j['trigger']}' | zip: '{os.path.basename(j['zip'])}'")
print("="*60)
print("問題なければ以降のセルを実行してください。\n")


## 2. 環境構築 (自動・一度実行すればスキップ)
Musubi Tuner の Qwen-Image 2.1 専用ブランチ（`nukuihayu/musubi-tuner:qwen-image-21`）をセットアップします。

In [ ]:
import os, sys

print("Python:", sys.version.split()[0])
if not os.path.exists("/content/.musubi_done"):
    if not os.path.exists("/content/musubi-tuner"):
        os.system("git clone -b qwen-image-21 https://github.com/nukuihayu/musubi-tuner.git /content/musubi-tuner")
        print("musubi-tuner cloned (branch: qwen-image-21)")
    else:
        os.system("cd /content/musubi-tuner && git pull")
        print("musubi-tuner updated")

    os.system("pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    os.system("pip install -q -e /content/musubi-tuner")
    os.system("pip install -q -U huggingface_hub accelerate transformers diffusers safetensors toml voluptuous einops bitsandbytes ftfy")
    os.system("pip cache purge")
    open("/content/.musubi_done", "w").write("done")
    print("Musubi Tuner 環境構築完了")
else:
    print("環境構築: 済み (スキップ)")

import torch
props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory/1e9:.1f}GB")

## 3. モデルダウンロード (自動・DL済みならスキップ)
Qwen-Image 2.1 のコンポーネントをダウンロードします。
- **DiT Transformer**: `qwen_image_2.1_bf16.safetensors` (~14 GB / Comfy-Org)
- **Text Encoder**: `qwen3vl_8b_bf16.safetensors` (~16 GB / Comfy-Org)
- **VAE**: `diffusion_pytorch_model.safetensors` (~1.35 GB / Qwen公式 Diffusers形式)
  ※ Musubi Tuner の VAE ローダーは Diffusers キー形式を要求するため公式重みを使用します
- **Configs & Processor**: `Qwen/Qwen-Image-2.1`

In [ ]:
import os, shutil, sys

try:
    from huggingface_hub import hf_hub_download, snapshot_download
except ImportError:
    os.system("pip install -q -U huggingface_hub")
    from huggingface_hub import hf_hub_download, snapshot_download

BASE_DIR = "/content/models/qwen_image_21"
os.makedirs(f"{BASE_DIR}/diffusion_models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/text_encoders", exist_ok=True)
os.makedirs(f"{BASE_DIR}/vae", exist_ok=True)

# 1. Configs & Processor (Qwen/Qwen-Image-2.1)
CONFIG_CHECK = f"{BASE_DIR}/processor/preprocessor_config.json"
if not os.path.exists(CONFIG_CHECK):
    print("Processor / Configs をダウンロード中 (Qwen/Qwen-Image-2.1)...")
    snapshot_download(
        "Qwen/Qwen-Image-2.1",
        allow_patterns=["processor/*", "*.json"],
        local_dir=BASE_DIR,
    )
    print("Processor DL完了")

# 2. VAE (0.68 GB)
# 旧形式(Comfy-Org版)のVAEが存在すれば削除
old_vae = f"{BASE_DIR}/vae/qwen_image_2.1_vae_bf16.safetensors"
if os.path.exists(old_vae):
    os.remove(old_vae)
    print("以前の非互換VAE (Comfy-Org形式) を削除しました")

# Musubi Tuner は Diffusers キー形式の VAE を要求します
VAE_PATH = f"{BASE_DIR}/vae/diffusion_pytorch_model.safetensors"
if not os.path.exists(VAE_PATH):
    print("VAE をダウンロード中 (Qwen/Qwen-Image-2.1 Diffusers形式)...\n")
    p = hf_hub_download("Qwen/Qwen-Image-2.1", "vae/diffusion_pytorch_model.safetensors")
    shutil.copy(p, VAE_PATH)
    print(f"VAE DL完了: {os.path.getsize(VAE_PATH)/1e9:.2f} GB")

# 3. Text Encoder (16 GB BF16)
TE_PATH = f"{BASE_DIR}/text_encoders/qwen3vl_8b_bf16.safetensors"
if not os.path.exists(TE_PATH):
    print("Text Encoder (BF16) をダウンロード中 (Comfy-Org/Qwen-Image-2.1)...")
    p = hf_hub_download("Comfy-Org/Qwen-Image-2.1", "text_encoders/qwen3vl_8b_bf16.safetensors")
    shutil.copy(p, TE_PATH)
    print(f"Text Encoder DL完了: {os.path.getsize(TE_PATH)/1e9:.2f} GB")

# 4. DiT Transformer (14 GB BF16)
DIT_PATH = f"{BASE_DIR}/diffusion_models/qwen_image_2.1_bf16.safetensors"
if not os.path.exists(DIT_PATH):
    print("DiT Transformer (BF16) をダウンロード中 (Comfy-Org/Qwen-Image-2.1)...")
    p = hf_hub_download("Comfy-Org/Qwen-Image-2.1", "diffusion_models/qwen_image_2.1_bf16.safetensors")
    shutil.copy(p, DIT_PATH)
    print(f"DiT Transformer DL完了: {os.path.getsize(DIT_PATH)/1e9:.2f} GB")

if os.path.isdir("/root/.cache/huggingface/hub"):
    os.system("rm -rf /root/.cache/huggingface/hub")
    print("HF一時キャッシュを削除してディスクを解放しました")

print("\n全モデル準備完了: MODELS_DONE")

## 4. 関数定義 (自動・定義するだけ)
データセット前処理、TOML生成、Latent/Text事前キャッシュ、Musubi学習、ComfyUI形式変換の各関数を定義します。

In [ ]:
import glob, os, re, shutil, subprocess, sys, zipfile

ENV = os.environ.copy()
ENV["PYTHONPATH"] = f"/content/musubi-tuner/src:{ENV.get('PYTHONPATH', '')}"

META_TAGS = {"character name", "character", "name", "symbols", "text", "english text",
             "dated", "score", "no humans", "commentary", "translated", "artist name"}

def calc_steps(n_imgs):
    if n_imgs <= 10:
        return 400
    if n_imgs <= 30:
        return 600
    if n_imgs <= 50:
        return 1000
    if n_imgs <= 100:
        return 1250
    if n_imgs <= 150:
        return 1500
    return 1800

def split_tags_and_prose(caption):
    parts = [p.strip() for p in caption.split(",") if p.strip()]
    prose_idx = None
    for i, p in enumerate(parts):
        words = p.split()
        if len(words) >= 4 and p[0].isupper() and not p.isupper() and not p[0].isdigit():
            prose_idx = i
            break
    if prose_idx is None:
        return parts, []
    return parts[:prose_idx], parts[prose_idx:]

def fix_caption(caption, trigger):
    tags, prose = split_tags_and_prose(caption)
    if trigger and trigger in tags:
        tags = [t for t in tags if t != trigger]
    tags = [t for t in tags if t.lower() not in META_TAGS]
    keep = []
    for t in tags:
        tlow = t.lower()
        if len(t.split()) == 1 and any(tlow in o.lower().split() and o.lower() != tlow for o in tags):
            continue
        keep.append(t)
    seen, deduped = set(), []
    for t in keep:
        if t.lower() not in seen:
            seen.add(t.lower())
            deduped.append(t)
    res = [trigger] + prose + deduped if trigger else prose + deduped
    return ", ".join(res)

def find_zip_path(zip_path):
    if os.path.exists(zip_path):
        return zip_path
    basename = os.path.basename(zip_path)
    scan_dirs = [
        "/content/drive/MyDrive/training/illust_sozai_zip",
        "/content/drive/MyDrive/qwen_image21",
        "/content/drive/MyDrive/qwen_image21/datasets",
        "/content/drive/MyDrive/datasets",
        "/content/drive/MyDrive",
        "/content",
    ]
    for d in scan_dirs:
        for cand in [os.path.join(d, zip_path), os.path.join(d, basename)]:
            if os.path.exists(cand):
                return cand
    return zip_path
def prepare_dataset(zip_path, trigger):
    resolved = find_zip_path(zip_path)
    assert os.path.exists(resolved), f"ZIPファイルが見つかりません: {zip_path}"

    os.system("rm -rf /content/dataset /content/cache")
    os.makedirs("/content/dataset", exist_ok=True)
    os.makedirs("/content/cache", exist_ok=True)

    with zipfile.ZipFile(resolved, "r") as z:
        z.extractall("/content/dataset")

    for f in glob.glob("/content/dataset/**/*", recursive=True):
        if os.path.isfile(f) and os.path.dirname(f) != "/content/dataset":
            shutil.move(f, os.path.join("/content/dataset", os.path.basename(f)))

    valid_exts = {".png", ".jpg", ".jpeg", ".webp"}
    imgs = sorted([
        os.path.join("/content/dataset", f) for f in os.listdir("/content/dataset")
        if os.path.isfile(os.path.join("/content/dataset", f)) and os.path.splitext(f)[1].lower() in valid_exts
    ])
    caps = sorted(glob.glob("/content/dataset/*.txt"))
    assert len(imgs) >= 1, f"ZIP内に有効な画像が見つかりません: {resolved}"

    changed = 0
    for txt in caps:
        raw = open(txt, encoding="utf-8", errors="ignore").read().strip()
        fixed = fix_caption(raw, trigger)
        if fixed != raw:
            changed += 1
        with open(txt, "w", encoding="utf-8") as f:
            f.write(fixed + "\n")

    print(f"データセット準備OK: 画像{len(imgs)}枚 / キャプション{len(caps)}個 (修正{changed}個) / trigger='{trigger}'")
    return len(imgs)

def make_dataset_toml(res=1024):
    res_val = int(res)
    toml = f'''[general]
resolution = [{res_val}, {res_val}]
caption_extension = ".txt"
batch_size = 1
enable_bucket = false
bucket_no_upscale = false

[[datasets]]
image_directory = "/content/dataset"
cache_directory = "/content/cache"
num_repeats = 1
'''
    toml_path = "/content/dataset.toml"
    with open(toml_path, "w", encoding="utf-8") as f:
        f.write(toml)
    print(f"データセットTOML作成OK: {toml_path} (単一解像度 {res_val}x{res_val})")
    return toml_path

def cache_latents(toml_path, vae_path):
    print("--- [1/3] Latents事前キャッシュ中... ---")
    cmd = [
        sys.executable, "/content/musubi-tuner/src/musubi_tuner/qwen_image_21_cache_latents.py",
        "--dataset_config", toml_path,
        "--vae", vae_path,
        "--vae_tiling",
    ]
    ret = subprocess.run(cmd, env=ENV, cwd="/content/musubi-tuner")
    if ret.returncode != 0:
        raise RuntimeError("Latents事前キャッシュが失敗しました。ログを確認してください。")
    print("Latents事前キャッシュ完了")

def cache_text_encoder(toml_path, te_path):
    print("--- [2/3] Text Encoder出力事前キャッシュ中... ---")
    cmd = [
        sys.executable, "/content/musubi-tuner/src/musubi_tuner/qwen_image_21_cache_text_encoder_outputs.py",
        "--dataset_config", toml_path,
        "--text_encoder", te_path,
        "--batch_size", "1",
        "--fp8_vl",
    ]
    ret = subprocess.run(cmd, env=ENV, cwd="/content/musubi-tuner")
    if ret.returncode != 0:
        raise RuntimeError("Text Encoder事前キャッシュが失敗しました。ログを確認してください。")
    print("Text Encoder事前キャッシュ完了")

def train_network(trigger, dit_path, toml_path, steps, lr, rank):
    print(f"--- [3/3] LoRA学習開始: {steps} steps / lr {lr} / rank {rank} ---")
    out_dir = f"/content/output/{trigger}"
    os.makedirs(out_dir, exist_ok=True)
    
    cmd = [
        "accelerate", "launch",
        "--num_cpu_threads_per_process", "1",
        "--mixed_precision", "bf16",
        "/content/musubi-tuner/src/musubi_tuner/qwen_image_21_train_network.py",
        "--dit", dit_path,
        "--dataset_config", toml_path,
        "--sdpa",
        "--mixed_precision", "bf16",
        "--gradient_checkpointing",
        "--network_module", "networks.lora_qwen_image_21",
        "--network_dim", str(rank),
        "--network_alpha", str(rank),
        "--optimizer_type", "AdamW",
        "--learning_rate", str(lr),
        "--lr_scheduler", "cosine_with_restarts",
        "--lr_scheduler_num_cycles", "2",
        "--timestep_sampling", "shift",
        "--discrete_flow_shift", "3.0",
        "--weighting_scheme", "none",
        "--max_train_steps", str(steps),
        "--save_every_n_steps", "100",
        "--output_dir", out_dir,
        "--output_name", trigger,
    ]
    ret = subprocess.run(cmd, env=ENV, cwd="/content/musubi-tuner")
    if ret.returncode != 0:
        raise RuntimeError("LoRA学習 (train_network) が失敗しました。ログを確認してください。")
    print("LoRA学習完了")

def convert_and_deliver(trigger, steps, lr, sweep_min=None):
    print("--- ComfyUI形式変換 & Google Drive納品中... ---")
    out_dir = OUTPUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    import datetime
    jst = datetime.timezone(datetime.timedelta(hours=9))
    dstr = datetime.datetime.now(jst).strftime("%Y%m%d")

    raw_dir = f"/content/output/{trigger}"
    ckpts = sorted(glob.glob(os.path.join(raw_dir, f"{trigger}*.safetensors")))
    assert ckpts, f"学習成果物 (.safetensors) が見つかりません: {raw_dir}"

    delivered = []
    for ckpt in ckpts:
        base = os.path.basename(ckpt)
        if "_comfy" in base:
            continue
        m = re.search(r"-step(\d+)\.safetensors$", base)
        if m:
            step_val = int(m.group(1))
            if sweep_min is not None and step_val < sweep_min:
                continue
            label = f"{step_val}steps"
        else:
            label = f"{steps}steps_final"

        # ComfyUI形式に変換
        comfy_local = os.path.join(raw_dir, f"{trigger}_{label}_comfy.safetensors")
        conv_cmd = [
            sys.executable, "-m", "musubi_tuner.convert_lora",
            "--input", ckpt,
            "--output", comfy_local,
            "--target", "other",
            "--diffusers_prefix", "transformer",
        ]
        conv_ret = subprocess.run(conv_cmd, env=ENV, cwd="/content/musubi-tuner")
        if conv_ret.returncode != 0:
            print(f"⚠️ 変換失敗: {base} (スキップ)")
            continue

        dst_name = f"{trigger}_qwen21_musubi_{dstr}_{label}_lr{lr}.safetensors"
        dst_path = os.path.join(out_dir, dst_name)
        shutil.copy(comfy_local, dst_path)
        delivered.append(dst_name)
        print(f"Drive保存: {dst_name} ({os.path.getsize(comfy_local)/1e6:.1f} MB)")

    print(f"納品完了: 計 {len(delivered)} 個のファイルを {out_dir} に保存しました。")
    return delivered

print("関数定義OK (prepare_dataset / make_dataset_toml / cache / train / convert_and_deliver)")

## 5. 全ジョブ実行 (自動・本丸)
JOBSの全データセットを順番に: **展開 → TOML作成 → キャッシュ → 学習 → ComfyUI変換 → Drive納品** します。

In [ ]:
import os, traceback, torch

FORCE_RERUN = True

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu} / VRAM: {vram_gb:.1f}GB")

res = RESOLUTION
DONE = "/content/jobs_done"
os.makedirs(DONE, exist_ok=True)

N = len(JOBS)
failed = []

for i, job in enumerate(JOBS, 1):
    trig, zip_ = job["trigger"], job["zip"]
    marker = os.path.join(DONE, f"{trig}.done")
    if os.path.exists(marker) and not FORCE_RERUN:
        print(f"\nSKIP: {trig} は完了済みです (再学習したい場合は FORCE_RERUN = True に設定してください)")
        continue

    print(f"\n{'='*64}\nJOB {i}/{N}: {trig} ({zip_.rsplit('/', 1)[-1]})\n{'='*64}")
    os.system(f"rm -rf /content/output/{trig}")

    try:
        n_imgs = prepare_dataset(zip_, trig)
        steps = job.get("steps") or STEPS or calc_steps(n_imgs)
        lr = job.get("lr") or LR
        rank = job.get("rank") or RANK
        print(f"→ ステップ数: {steps} (画像 {n_imgs} 枚) / lr: {lr} / rank: {rank}")

        toml_path = make_dataset_toml(res)
        cache_latents(toml_path, VAE_PATH)
        cache_text_encoder(toml_path, TE_PATH)
        train_network(trig, DIT_PATH, toml_path, steps, lr, rank)
        convert_and_deliver(trig, steps, lr, sweep_min=job.get("sweep_min"))

        with open(marker, "w") as f:
            f.write("done")
        print(f"JOB_DONE: {trig} ({i}/{N})")
    except Exception as e:
        print(f"❌ エラー発生: {trig}: {e}")
        traceback.print_exc()
        failed.append(trig)

print("\n" + "="*64)
if failed:
    print(f"⚠️ 失敗したジョブがあります: {failed}")
else:
    print("🎉 全ジョブが正常に完了しました！")
print("="*64)

## 完了! ComfyUIでの利用手順

1. **出力ファイル**:
   - Google Drive の `MyDrive/qwen_image21/outputs/` に `..._comfy...safetensors` として保存されています。
2. **ComfyUIでの接続**:
   - 生成された `.safetensors` を `ComfyUI/models/loras/` に配置します。
   - `LoraLoader` ノードで選択し、Qwen-Image 2.1 の DiT モデルに接続します（`convert_lora` により ComfyUI ネイティブ形式に変換済みのため、警告なしでロードされます）。
   - プロンプトにトリガーワード（例: `kibisake_style`）を含めて生成してください。